# 👑 DropQueen
## Notebook 5: Docker & CI/CD Deployment
**Project:** AI-Powered Product Demand & Sales Forecasting Engine for TikTok Shop & Amazon  
**Author:** Chastity Lewis  
**Course:** CISC 610 — DevOps and MLOps | Mercy University | Spring 2026  

---

### 📌 Notebook Goals
1. Generate all Docker deployment files (`Dockerfile`, `requirements.txt`, `docker-compose.yml`)
2. Generate the GitHub Actions CI/CD pipeline (`.github/workflows/deploy.yml`)
3. Generate the MLflow model tracking setup
4. Verify the full project file structure
5. Download all deployment files as a zip

> ⚠️ Note: Docker commands run locally on your machine, not inside Colab.
> This notebook generates all the files you need and explains how to deploy them.

---

## Step 1: Install & Import Libraries

In [ ]:
import os
import zipfile
from google.colab import files

# Create project folder structure
os.makedirs('dropqueen/models', exist_ok=True)
os.makedirs('dropqueen/.github/workflows', exist_ok=True)
os.makedirs('dropqueen/mlflow', exist_ok=True)

print('✅ Libraries loaded!')
print('✅ Project folder structure created!')
print()
print('📁 dropqueen/')
print('   ├── models/')
print('   ├── .github/workflows/')
print('   └── mlflow/')

## Step 2: Generate Dockerfile

In [ ]:
dockerfile = '''# ── DropQueen Flask API — Dockerfile ──────────────────────────────────────────
# Base image
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PORT=5000

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for Docker layer caching)
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application files
COPY app.py .
COPY models/ ./models/

# Copy model files
COPY demand_model.pkl .
COPY engagement_model.pkl .
COPY model_config.json .

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:5000/health || exit 1

# Run the application
CMD ["python", "app.py"]
'''

with open('dropqueen/Dockerfile', 'w') as f:
    f.write(dockerfile)

print('✅ Dockerfile created!')
print()
print(dockerfile)

## Step 3: Generate requirements.txt

In [ ]:
requirements = '''# DropQueen API — Python Dependencies

# Web Framework
flask==3.0.0
gunicorn==21.2.0

# Machine Learning
scikit-learn==1.4.0
joblib==1.3.2

# Data Processing
pandas==2.1.4
numpy==1.26.3

# Model Tracking
mlflow==2.10.0

# Utilities
python-dotenv==1.0.0
'''

with open('dropqueen/requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ requirements.txt created!')
print()
print(requirements)

## Step 4: Generate docker-compose.yml

In [ ]:
docker_compose = '''# DropQueen — Docker Compose
version: "3.8"

services:

  # ── Flask API ──────────────────────────────────────────────────────────────
  dropqueen-api:
    build: .
    container_name: dropqueen-api
    ports:
      - "5000:5000"
    environment:
      - PORT=5000
      - MLFLOW_TRACKING_URI=http://mlflow:5001
    volumes:
      - ./models:/app/models
      - ./demand_model.pkl:/app/demand_model.pkl
      - ./engagement_model.pkl:/app/engagement_model.pkl
      - ./model_config.json:/app/model_config.json
    depends_on:
      - mlflow
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:5000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # ── MLflow Tracking Server ────────────────────────────────────────────────
  mlflow:
    image: python:3.11-slim
    container_name: dropqueen-mlflow
    ports:
      - "5001:5001"
    command: >
      bash -c "pip install mlflow && mlflow server
      --host 0.0.0.0
      --port 5001
      --backend-store-uri sqlite:///mlflow.db
      --default-artifact-root ./mlruns"
    volumes:
      - mlflow-data:/mlflow
    restart: unless-stopped

volumes:
  mlflow-data:
'''

with open('dropqueen/docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print('✅ docker-compose.yml created!')
print()
print(docker_compose)

## Step 5: Generate GitHub Actions CI/CD Pipeline

In [ ]:
github_actions = '''# ── DropQueen CI/CD Pipeline — GitHub Actions ─────────────────────────────────
name: DropQueen CI/CD Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  schedule:
    # Auto-retrain every Sunday at midnight
    - cron: "0 0 * * 0"

jobs:

  # ── Job 1: Test ─────────────────────────────────────────────────────────────
  test:
    name: Run Tests
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install pytest

      - name: Run API health check test
        run: |
          python -c "
          import joblib, json
          demand_model = joblib.load('demand_model.pkl')
          engagement_model = joblib.load('engagement_model.pkl')
          with open('model_config.json') as f:
              config = json.load(f)
          print('✅ Models loaded successfully')
          print('✅ Config loaded successfully')
          "

  # ── Job 2: Build & Push Docker Image ────────────────────────────────────────
  build:
    name: Build Docker Image
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == \'refs/heads/main\'

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Log in to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}

      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/dropqueen-api:latest
            ${{ secrets.DOCKER_USERNAME }}/dropqueen-api:${{ github.sha }}

  # ── Job 3: Auto-Retrain Models ───────────────────────────────────────────────
  retrain:
    name: Auto-Retrain Models
    runs-on: ubuntu-latest
    if: github.event_name == \'schedule\'

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install -r requirements.txt

      - name: Retrain models
        run: |
          echo "🔄 Retraining DropQueen models..."
          python retrain.py
          echo "✅ Models retrained successfully"

      - name: Commit updated models
        run: |
          git config --local user.email "action@github.com"
          git config --local user.name "GitHub Actions"
          git add demand_model.pkl engagement_model.pkl model_config.json
          git commit -m "🤖 Auto-retrain: updated models $(date +%Y-%m-%d)" || echo "No changes"
          git push
'''

with open('dropqueen/.github/workflows/deploy.yml', 'w') as f:
    f.write(github_actions)

print('✅ GitHub Actions CI/CD pipeline created!')
print()
print(github_actions)

## Step 6: Generate MLflow Tracking Setup

In [ ]:
mlflow_setup = '''# ── DropQueen MLflow Model Tracking ───────────────────────────────────────────
"""
Run this script to log your trained models to MLflow.
Usage: python mlflow/track_models.py
"""

import mlflow
import mlflow.sklearn
import joblib
import json
from datetime import datetime

# ── Load models and config ────────────────────────────────────────────────────
demand_model     = joblib.load("demand_model.pkl")
engagement_model = joblib.load("engagement_model.pkl")

with open("model_config.json", "r") as f:
    config = json.load(f)

# ── Set MLflow tracking URI ───────────────────────────────────────────────────
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("DropQueen")

# ── Log Demand Model ──────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f"demand_model_{datetime.now().strftime(\'%Y%m%d\')}"):
    mlflow.log_param("model_type",  type(demand_model).__name__)
    mlflow.log_param("features",    config["demand_features"])
    mlflow.log_param("target",      "demand_spike")
    mlflow.log_metric("accuracy",   config["demand_accuracy"])
    mlflow.log_metric("f1_score",   config["demand_f1"])
    mlflow.sklearn.log_model(demand_model, "demand_model")
    print("✅ Demand model logged to MLflow")

# ── Log Engagement Model ──────────────────────────────────────────────────────
with mlflow.start_run(run_name=f"engagement_model_{datetime.now().strftime(\'%Y%m%d\')}"):
    mlflow.log_param("model_type",  type(engagement_model).__name__)
    mlflow.log_param("features",    config["tiktok_features"])
    mlflow.log_param("target",      "engagement_level")
    mlflow.log_metric("accuracy",   config["tiktok_accuracy"])
    mlflow.log_metric("f1_score",   config["tiktok_f1"])
    mlflow.sklearn.log_model(engagement_model, "engagement_model")
    print("✅ Engagement model logged to MLflow")

print()
print("🚀 Both models tracked in MLflow!")
print("   View at: http://localhost:5001")
'''

with open('dropqueen/mlflow/track_models.py', 'w') as f:
    f.write(mlflow_setup)

print('✅ MLflow tracking script created!')

## Step 7: Generate .gitignore & README

In [ ]:
gitignore = '''# Python
__pycache__/
*.py[cod]
*.egg-info/
.env
venv/

# Data
*.csv
data/

# MLflow
mlruns/
mlflow.db

# Docker
.dockerignore

# Jupyter
.ipynb_checkpoints/
'''

readme = '''# 👑 DropQueen — AI-Powered Demand & Sales Forecasting Engine

**Course:** CISC 610 — DevOps and MLOps | Mercy University | Spring 2026  
**Author:** Chastity Lewis

---

## 🚀 Quick Start

### Run with Docker
```bash
# Build and start all services
docker-compose up --build

# API will be live at:
# http://localhost:5000

# MLflow UI will be live at:
# http://localhost:5001
```

### Run locally
```bash
pip install -r requirements.txt
python app.py
```

---

## 📡 API Endpoints

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | /health | Health check |
| POST | /predict/demand | Predict product demand spike |
| POST | /predict/trend | Predict TikTok engagement level |
| GET | /products/top | Get top predicted drops |
| GET | /models/versions | List model metadata |

---

## 🧠 Models
- **Demand Spike Model** — Random Forest (87% accuracy)
- **TikTok Engagement Model** — Logistic Regression

---

## 🛠️ Tech Stack
Python · Flask · scikit-learn · Docker · GitHub Actions · MLflow · AWS
'''

with open('dropqueen/.gitignore', 'w') as f:
    f.write(gitignore)

with open('dropqueen/README.md', 'w') as f:
    f.write(readme)

print('✅ .gitignore created!')
print('✅ README.md created!')

## Step 8: Verify Project Structure

In [ ]:
print('📁 DropQueen Project Structure:')
print()

for root, dirs, files_list in os.walk('dropqueen'):
    # Skip hidden folders from display
    level = root.replace('dropqueen', '').count(os.sep)
    indent = '   ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '   ' * (level + 1)
    for file in files_list:
        print(f'{subindent}📄 {file}')

print()
print('✅ All deployment files generated!')

## Step 9: Download All Files as ZIP

In [ ]:
# Create zip of all deployment files
zip_filename = 'dropqueen_deployment.zip'

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk('dropqueen'):
        for file in files_list:
            file_path = os.path.join(root, file)
            zipf.write(file_path)

print(f'✅ ZIP created: {zip_filename}')
print()

# Download the zip
files.download(zip_filename)
print('📥 Downloading dropqueen_deployment.zip...')

## Step 10: Deployment Instructions

In [ ]:
print('=' * 60)
print('   👑 DropQueen — Deployment Instructions')
print('=' * 60)
print()
print('📦 STEP 1: Unzip dropqueen_deployment.zip on your computer')
print()
print('📁 STEP 2: Copy your model files into the folder:')
print('   - demand_model.pkl')
print('   - engagement_model.pkl')
print('   - model_config.json')
print('   - app.py  (from Notebook 4)')
print()
print('🐳 STEP 3: Run Docker:')
print('   docker-compose up --build')
print()
print('🌐 STEP 4: Test your live API:')
print('   http://localhost:5000/health')
print()
print('📊 STEP 5: View MLflow dashboard:')
print('   http://localhost:5001')
print()
print('🔁 STEP 6: Push to GitHub for CI/CD:')
print('   git add .')
print('   git commit -m "Add Docker & CI/CD deployment"')
print('   git push origin main')
print()
print('=' * 60)
print('✅ Notebook 5 Complete! DropQueen is ready to deploy! 🚀')
print('=' * 60)